# Fib (Java)

[![Apri in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap14_ricorsione/Fib_java.ipynb)

Capitolo 14 — Ricorsione. Notebook eseguibile di `Fib`: l'algoritmo in Java, eseguito sugli esempi del capitolo, con la cella «Prova tu» per provarlo con dati tuoi. Lo pseudocodice, gli esempi e le dimostrazioni sono nella [pagina del capitolo](https://fabiofurini.github.io/algoritmi/ricorsione/14-ricorsione/).

Lo stesso notebook in [Python](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap14_ricorsione/Fib_python.ipynb) e [C++](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap14_ricorsione/Fib_cpp.ipynb); tutti gli algoritmi del capitolo nel [notebook del capitolo](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap14_ricorsione_java.ipynb).

**Come si esegue.** Clic sul badge «Apri in Colab», poi *Runtime → Esegui tutto* (Ctrl+F9), oppure una cella alla volta con Maiusc+Invio, dall'alto in basso. La prima cella verifica che il JDK ci sia e, se manca, lo installa. Ogni programma Java è scritto su un file dalla cella `%%writefile`, poi compilato con `javac` ed eseguito con `java` nella cella successiva.

**Nomi.** In questo capitolo non ci sono array: variabili e parametri hanno gli stessi nomi dello pseudocodice. Lo stack e i contatori sono campi `static` della classe, e la traccia si attiva con `trace = true`.

## Preparazione

Su Colab il JDK di solito c'è già: la cella qui sotto lo verifica e, se manca, lo installa.

In [1]:
!javac -version || (apt-get -qq update && apt-get -qq install -y default-jdk-headless)

javac 25.0.4.1


## Numeri di Fibonacci: Fib

**Input:** un numero naturale $n$. **Output:** $F(n)$, con $F(0) = 0$, $F(1) = 1$ e $F(n) = F(n-1) + F(n-2)$. Algoritmo ricorsivo naive: $C(n) = 2F(n+1) - 1$ chiamate e $S(n) = 4F(n+1) - 2 = 2^{\Theta(n)}$ passi.

Pseudocodice: sezione 14.5 della [pagina del capitolo](https://fabiofurini.github.io/algoritmi/ricorsione/14-ricorsione/).

In [2]:
%%writefile Fib.java
public class Fib
{
static long[] cnt = new long[20];   // cnt[r] = numero di esecuzioni della riga r (numerazione delle dispense)
static long calls = 0;              // numero di chiamate = nodi dell'albero delle chiamate
static int maxDepth = 0;            // profondita' della ricorsione = altezza massima dello stack
static long mults = 0;              // moltiplicazioni (FastPower)
static long moves = 0;              // mosse (Hanoi)
static int ev = 0;                  // contatore degli eventi della traccia
static java.util.ArrayList<Integer> stack = new java.util.ArrayList<>();   // lo stack delle chiamate: gli argomenti delle chiamate attive
static long[] seen = new long[64];  // seen[x] = numero di chiamate con argomento x
static boolean trace = false;       // true: stampa la traccia dell'esecuzione

static void reset()
{
    java.util.Arrays.fill(cnt, 0);
    java.util.Arrays.fill(seen, 0);
    calls = 0; maxDepth = 0; mults = 0; moves = 0; ev = 0;
    stack.clear();
}

static long total()
{
    long s = 0;
    for (long c : cnt) s += c;
    return s;
}

static void printSteps(int... lines)
{
    System.out.println("Riga | Esecuzioni");
    for (int r : lines) System.out.println(String.format("%4d | %10d", r, cnt[r]));
    System.out.println("Passi totali S = " + total());
}

static void push(int x)             // inizia una chiamata: il suo frame va in cima allo stack
{
    calls++;
    stack.add(x);
    if (stack.size() > maxDepth) maxDepth = stack.size();
    if (x >= 0 && x < 64) seen[x]++;
}

static void pop()                   // la chiamata termina: il suo frame esce dallo stack
{
    stack.remove(stack.size() - 1);
}

static void event(String text)      // riga numerata della traccia, con lo stack
{
    ev++;
    System.out.println(String.format("%2d. %s   stack: %s", ev, text, stack));
}

static void say(String call, String action)   // riga della traccia: la chiamata, rientrata di un livello per ogni chiamata attiva
{
    System.out.println(String.format("%-28s| %s", "  ".repeat(stack.size() - 1) + call, action));
}

    // Fib(n)
    static long fib(int n)
    {
        push(n); cnt[2]++;
        if (n <= 1)
        {
            cnt[3]++;
            if (trace) say(String.format("Fib(%d)", n), String.format("caso base, restituisce %d", n));
            pop();
            return n;
        }
        if (trace) say(String.format("Fib(%d)", n), String.format("chiama Fib(%d)", n - 1));
        long x = fib(n - 1);                       // le due chiamate sono separate solo per stampare la traccia
        if (trace) say(String.format("Fib(%d)", n), String.format("chiama Fib(%d)", n - 2));
        long y = fib(n - 2);
        cnt[5]++;
        if (trace) say(String.format("Fib(%d)", n), String.format("restituisce %d + %d = %d", x, y, x + y));
        pop();
        return x + y;
    }

    static long fibValue(int k)                    // F(k) con un ciclo, per controllare
    {
        long a = 0, b = 1;
        for (int i = 0; i < k; i++)
        {
            long t = a + b;
            a = b;
            b = t;
        }
        return a;
    }

    public static void main(String[] args)
    {
        // Esempio delle dispense: l'esecuzione di Fib(4), in ordine di visita in profondita'
        int n = 4;
        reset();
        trace = true;
        System.out.println(String.format("%-28s| %s", "chiamata", "azione"));
        long r = fib(n);
        trace = false;
        System.out.printf("F(%d) = %d, chiamate C(%d) = %d = 2F(%d) - 1, profondita' = %d\n", n, r, n, calls, n + 1, maxDepth);

        // Fib(5): gli stessi sottoproblemi sono risolti piu' volte
        System.out.printf("\n");
        n = 5;
        reset();
        r = fib(n);
        System.out.printf("Fib(%d) = %d: %d chiamate, profondita' %d, S = %d passi\n", n, r, calls, maxDepth, total());
        System.out.printf("  k | chiamate di Fib(k)\n");
        for (int k = n; k >= 0; k--) System.out.printf("%3d | %d\n", k, seen[k]);
        printSteps(2, 3, 5);

        // Numero di chiamate e di passi: C(n) = 2F(n+1) - 1 e S(n) = 4F(n+1) - 2
        System.out.printf("\n");
        System.out.printf("  n |   F(n) |     C(n) | 2F(n+1) - 1 |     S(n) | 4F(n+1) - 2 | profondita'\n");
        int[] values = {0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 20, 30};
        for (int v : values)
        {
            reset();
            r = fib(v);
            System.out.printf("%3d | %6d | %8d | %11d | %8d | %11d | %d\n", v, r, calls, 2 * fibValue(v + 1) - 1, total(), 4 * fibValue(v + 1) - 2, maxDepth);
        }
    }
}

Writing Fib.java


In [3]:
!javac Fib.java && java Fib

chiamata                    | azione
Fib(4)                      | chiama Fib(3)
  Fib(3)                    | chiama Fib(2)
    Fib(2)                  | chiama Fib(1)
      Fib(1)                | caso base, restituisce 1
    Fib(2)                  | chiama Fib(0)
      Fib(0)                | caso base, restituisce 0
    Fib(2)                  | restituisce 1 + 0 = 1
  Fib(3)                    | chiama Fib(1)
    Fib(1)                  | caso base, restituisce 1
  Fib(3)                    | restituisce 1 + 1 = 2
Fib(4)                      | chiama Fib(2)
  Fib(2)                    | chiama Fib(1)
    Fib(1)                  | caso base, restituisce 1
  Fib(2)                    | chiama Fib(0)
    Fib(0)                  | caso base, restituisce 0
  Fib(2)                    | restituisce 1 + 0 = 1
Fib(4)                      | restituisce 2 + 1 = 3
F(4) = 3, chiamate C(4) = 9 = 2F(5) - 1, profondita' = 4

Fib(5) = 5: 15 chiamate, profondita' 5, S = 30 passi
  k | chiamate di

**Prova tu.** Cambia $n$ e guarda quanto crescono le chiamate.

In [4]:
%%writefile ProvaFib.java
public class ProvaFib
{
    public static void main(String[] args)
    {
        int n = 6;        // <-- cambia n (con n > 45 ci vuole un po')
        Fib.reset();
        long r = Fib.fib(n);
        System.out.printf("F(%d) = %d, chiamate = %d, S = %d\n", n, r, Fib.calls, Fib.total());
    }
}

Writing ProvaFib.java


In [5]:
!javac ProvaFib.java && java ProvaFib

F(6) = 8, chiamate = 25, S = 50


---

Materiale didattico di [Fabio Furini](https://fabiofurini.github.io/) — DIAG, Sapienza Università di Roma.
Testo, figure e dati [CC BY 4.0](https://github.com/fabiofurini/algoritmi/blob/main/LICENSE), codice [MIT](https://github.com/fabiofurini/algoritmi/blob/main/LICENSE-CODE).